# Pratica - Bot de Cadastro no Windows
## com PyAutoGUI (UI) e Pandas (CSV) -> Parte 1

Objetivos:
1) Ler um CSV (em lote) com 20 cadastros (pandas);
2) Abrir o portal no navegador e cadastrar alguns via UI (PyAutoGUI);
3) Cadastrar o lote inteiro via DOM (Playwright);
4) Validar via DOM (quantidade e busca por CPF);
5) Opcional: por o robo em produção via FastAPI para rodar como serviço no Windows.

Dicas:
- Enquanto PyAutoGUI roda: não encoste no PC;
- Fazer um FAILSAFE: mouse no canto superior esquerdo aborta.

---

## 0) Instalacao (Windows)

Terminal:

```bash
python -m venv .rpa
.rpa\Scripts\activate

python -m pip install -U pip setuptools wheel
pip install -U greenlet
pip install -U playwright
python -m playwright install

pip install pyautogui rpa pandas fastapi uvicorn
```
Para instalar o greenlet, precisa-se instalar previamente o Microsoft Visual C++ 14.0 Redist (ou superior):

```bash
vs_buildtools.exe --norestart --passive --downloadThenInstall --includeRecommended --add Microsoft.VisualStudio.Workload.NativeDesktop --add Microsoft.VisualStudio.Workload.VCTools --add Microsoft.VisualStudio.Workload.MSBuildTools
```

Para o VSCode reconhecer o Virtual Environment criado (dentro do venv criado):

```bash
pip install ipykernel
python -m ipykernel install --user --name "rpa" --display-name "Python (.rpa)"
```

Arquivos:
- Portal: `portal_fake` (index.html + styles.css + app.js)
- CSV: `cadastros_portal_fake_20.csv`

In [19]:
import os
import pandas as pd
from pathlib import Path

---

## 1) Configurar caminhos

Edite `PORTAL_INDEX` apontando para o `index.html` do portal extraido.

In [ ]:
# PORTAL_INDEX = r"C:\CAMINHO\PARA\portal_fake\index.html"
PORTAL_INDEX = r"C:\Users\Turma01\Documents\messyas\ax-lg\01-py_auto_gui\desafios\portal-fake\pagina\index.html"

CSV_PATH = r"C:\Users\Turma01\Documents\messyas\ax-lg\01-py_auto_gui\desafios\portal-fake\cadastros_portal_fake_20.xlsx"

def as_file_url(path_str: str) -> str:
    return Path(path_str).resolve().as_uri()

portal_url = as_file_url(PORTAL_INDEX)
portal_url

'file:///C:/Users/Turma01/Documents/messyas/ax-lg/01-py_auto_gui/desafios/portal-fake/pagina/index.html'

## Pandas (dados/planilhas)

O Pandas é a ferramenta padrão pra trabalhar com **tabelas** (CSV/Excel).  
No RPA, ele é o “motor do lote”: você lê uma planilha, vira uma lista de registros e automatiza em loop.

**Quando usar:**
- ler CSV/Excel com cadastros;
- limpar/validar dados (CPF vazio, email inválido, etc.);
- gerar relatórios (CSV final com OK/ERRO).

**Resumo dos comandos mais usados:**
- `pd.read_csv("arquivo.csv")` → lê CSV;
- `df.head()` → prévia;
- `df.fillna("")` → tratar vazios;
- `df.to_dict(orient="records")` → lista de dicts (perfeito pra loop do robô);
- `df.to_csv("saida.csv", index=False)` → exportar relatório.

In [21]:
import pandas as pd

df = pd.read_csv("cadastros_portal_fake_20.csv").fillna("")
print(df.head(3))

# Transformar em lista de registros (um dict por linha)
records = df.to_dict(orient="records")
print(records[0])

# Gerar relatório simples
rel = df[["cpf","email"]].copy()
rel["status_exec"] = "PENDENTE"
rel.to_csv("relatorio_exemplo.csv", index=False, encoding="utf-8")

   id_solicitacao   nome sobrenome          cpf                        email  \
0               1    Ana     Silva  10000012482      ana.silva.1@exemplo.com   
1               2  Bruno     Rocha  10000012619    bruno.rocha.2@exemplo.com   
2               3  Carla   Almeida  10000012756  carla.almeida.3@exemplo.com   

          telefone  nascimento     status  \
0  (92) 99001-1001  1988-01-01      ATIVO   
1  (92) 99002-1002  1989-01-17   PENDENTE   
2  (92) 99003-1003  1990-02-03  BLOQUEADO   

                                 endereco               observacao  \
0  Rua A, 100 - Bairro Centro - Manaus/AM             Cadastro OK.   
1  Rua B, 107 - Bairro Aleixo - Manaus/AM  Aguardando confirmacao.   
2  Rua C, 114 - Bairro Flores - Manaus/AM       Revisar documento.   

  solicitante prioridade       canal  
0    Equipe A      BAIXA       email  
1    Equipe B      MEDIA    whatsapp  
2    Equipe C       ALTA  presencial  
{'id_solicitacao': 1, 'nome': 'Ana', 'sobrenome': 'Silva', 'c

---

## 2) Ler o CSV com pandas

In [22]:
CSV_PATH = "cadastros_portal_fake_20.csv"
df = pd.read_csv(CSV_PATH).fillna("")
required_cols = ["nome","sobrenome","cpf","email","telefone","nascimento","status","endereco","observacao"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV faltando colunas: {missing}")

print("Total registros:", len(df))
df.head()

Total registros: 20


,id_solicitacao,nome,sobrenome,cpf,email,telefone,nascimento,status,endereco,observacao,solicitante,prioridade,canal
0,1,Ana,Silva,10000012482,ana.silva.1@exemplo.com,(92) 99001-1001,1988-01-01,ATIVO,"Rua A, 100 - Bairro Centro - Manaus/AM",Cadastro OK.,Equipe A,BAIXA,email
1,2,Bruno,Rocha,10000012619,bruno.rocha.2@exemplo.com,(92) 99002-1002,1989-01-17,PENDENTE,"Rua B, 107 - Bairro Aleixo - Manaus/AM",Aguardando confirmacao.,Equipe B,MEDIA,whatsapp
2,3,Carla,Almeida,10000012756,carla.almeida.3@exemplo.com,(92) 99003-1003,1990-02-03,BLOQUEADO,"Rua C, 114 - Bairro Flores - Manaus/AM",Revisar documento.,Equipe C,ALTA,presencial
3,4,Diego,Gomes,10000012893,diego.gomes.4@exemplo.com,(92) 99004-1004,1991-02-20,ATIVO,"Rua D, 121 - Bairro Parque 10 - Manaus/AM",Contato pendente.,Equipe D,BAIXA,email
4,5,Evelyn,Pereira,10000013030,evelyn.pereira.5@exemplo.com,(92) 99005-1005,1992-03-08,PENDENTE,"Rua E, 128 - Bairro Adrianopolis - Manaus/AM",Bloqueio temporario.,Equipe E,MEDIA,whatsapp


## PyAutoGUI (automação por tela)

O PyAutoGUI é “controle remoto do seu mouse e teclado”.  
Ele **não entende DOM**, não sabe o que é botão/campo HTML. Ele só vê **a tela** e executa ações: mover mouse, clicar, digitar, apertar teclas, printar, etc.

**Quando usar:**
- app legado/desktop (ERP, Notepad, sistemas antigos);
- janela fora do navegador;
- “qualquer coisa que aparece na tela”, mesmo que não tenha API.

**Quando NÃO usar:**
- quando você precisa de estabilidade e elementos web (DOM). Aí Playwright/Selenium é melhor.

**Resumo dos comandos mais usados:**
- `pg.position()` → pega a posição atual do mouse (x,y);
- `pg.moveTo(x,y)` / `pg.click(x,y)` → move e clica;
- `pg.write("texto")` → digita;
- `pg.press("tab")` / `pg.hotkey("ctrl","s")` → teclas e atalhos;
- `pg.PAUSE` e `time.sleep()` → controla o ritmo;
- `pg.FAILSAFE=True` → canto superior esquerdo aborta.

---

## 3) Desenvolver o Bot

# PARTE A.1 — PyAutoGUI (UI)

Vamos cadastrar os dados dos usuários usando apenas o PyAutoGUI, para isso precisamos:
- apagar cadastros anteriores para evitar conflitos;
- abrir o portal no navegador padrão;
- navegar ate o botão "Novo cadastro" via TAB (ajustável);
- preencher via TAB/digitação;
- salvar e repetir.

In [29]:
# apagar tudo via UI (PyAutoGUI) por TAB (sem coordenadas)
import pyautogui as pg
import time 

pg.PAUSE = 0.02
pg.FAILSAFE = True

In [35]:
TABS_ZERAR = 4

def open_portal(url: str):
    #pra acessar pelo arquivo dentro do sistema
    os.startfile(url)
    time.sleep(3)

def zerar_base():
    for _ in range(TABS_ZERAR):
        pg.press("tab")
        time.sleep(0.1)
    pg.press("enter")
    time.sleep(0.5)
    pg.press("enter")
    time.sleep(0.5)
    
open_portal(portal_url)
zerar_base()

In [47]:
# realizar cadastro via UI (PyAutoGUI) por TAB (sem coordenadas)
PAGE_LOAD = 0.3
OPEN_MODAL = 1.0
SAVE_REG = 0.5

TABS = 1

def open_portal(url: str):
    #pra acessar pelo arquivo dentro do sistema
    os.startfile(url)
    time.sleep(3)

def open_modal():
    for _ in range(TABS):
        pg.press("tab")
        time.sleep(0.1)
    pg.press("enter")
    time.sleep(OPEN_MODAL)
    
def preencher_cadastro(r:dict):
    pg.write(str(r["nome"]), interval=0.03); pg.press('tab')
    pg.write(str(r["sobrenome"]), interval=0.03); pg.press('tab')
    pg.write(str(r["cpf"]), interval=0.03); pg.press('tab')
    pg.write(str(r["email"]), interval=0.03); pg.press('tab')
    pg.write(str(r["telefone"]), interval=0.03); pg.press('tab')
    pg.write(str(r["nascimento"]), interval=0.03); pg.press('tab'); pg.press('tab')
    #no caso do status, nos temos um combobox, onde nao tem escrita e sim movimento por setas.
    st = str(r.get('status', 'ATIVO')).upper().strip()
    if st == "PENDENTE":
        pg.press('down')
    elif st ==  "BLOQUEADO":
        pg.press('down')
        pg.press('down')
    pg.press('tab')
    pg.time.sleep(0.3)
             
    pg.write(str(r["observacao"]), interval=0.03); pg.press('tab')
    pg.write(str(r["endereco"]), interval=0.03); pg.press('tab'); pg.press('tab')
    pg.press('enter')
    time.sleep(SAVE_REG)
    
def cadastrar_lote(df, limit):
    open_portal(portal_url)
    out = []
    for i, r in df.head(limit).iterrows():
        d = r.to_dict()
        try:
            open_modal()
            preencher_cadastro(d)
            out.append({'id': d.get('id_solicitacao', i+1), 'cpf': d['cpf'], 'status': 'SUCESSO'})
        except Exception as e:
            print(f"Erro no cadastro do CPF {d['cpf']}: {e}")
            out.append({'id': d.get('id_solicitacao', i+1), 'cpf': d['cpf'], 'status': f'FALHA: {e}'})
    return pd.DataFrame(out)
            
            

In [ ]:
open_portal(portal_url)
zerar_base()

In [52]:
report = cadastrar_lote(df, limit=3)
report

,id,cpf,status
0,1,10000012482,SUCESSO
1,2,10000012619,SUCESSO
2,3,10000012756,SUCESSO


### A1) Rodar teste curto (3 cadastros)

Depois aumente para 20.

In [ ]:
# testar o bot


# PARTE A.2 — PyAutoGUI (UI) com mouse (moveTo + click)

Agora vamos fazer **o mesmo cadastro**, só que usando **mouse por coordenadas**:
- mover o mouse lentamente até a posição do botão/campo;
- clicar;
- digitar;
- repetir para cada campo.

Por que isso existe na prática de RPA?
- porque em ambiente real você vai pegar sistemas que não dão DOM/API;
- e aí sobra “clicar onde aparece”.

⚠️ Importante:
- coordenada muda com resolução/zoom/posição da janela;
- então a gente precisa **calibrar** (capturar as posições).

In [51]:
# Apagar tudo via UI (PyAutoGUI) por clique (coordenadas)
import pyautogui as pg
import time

def zerar_base_mouse():
    pg.moveTo(1179, 310, duration=0.5)
    pg.click()
    time.sleep(0.5)
    pg.press("enter")

open_portal(portal_url)
zerar_base_mouse()

In [66]:
# ver as coordenadas
def pegar_coordenadas(tempo=2):
    for i in range(tempo, 0, -1): 
        print(i)
        time.sleep(1)
    x, y = pg.position()
    print(f"posição: {x}, {y}")	
    return (x, y)

#Coordenadas de zerar a base (1179, 310)
# novo = (710, 335)
# nome = (558, 320)
# sobrenome = (226, 598)
# cpf = (377, 748)
# email = (399, 751)
# telefone = (288, 886)
# nascimento = (408, 1030)
# status = (207, 1181)
# observacao = (344, 1327)
# endereco = (626, 1274)
# salvar = (1149, 1512)

pegar_coordenadas()

2
1
posição: 1149, 1512


(1149, 1512)

## Cadastro por mouse (coordenadas)

Depois de calibrar, cole as coordenadas no dicionário `POS`.
Aí o robô:
1) clica no botão "Novo cadastro";
2) preenche cada campo clicando nele;
3) no status: clica no select e usa setas;
4) clica em salvar.

In [69]:
# realizar o cadastro via UI (PyAutoGUI) por clique (coordenadas)
pos = {
    "btn_novo": (710, 335),
    "nome": (558, 320),
    "sobrenome": (226, 598),
    "cpf": (377, 748),
    "email": (399, 751),
    "telefone": (288, 886),
    "nascimento": (408, 1030),
    "status": (207, 1181),
    "observacao": (344, 1327),
    "endereco": (626, 1274),
    "btn_salvar": (1149, 1512)
}

def mover_mouse(key):
    x, y = pos[key]
    pg.moveTo(x, y, duration=0.2)
    pg.click()
    
def limpar_campos_e_escrever(texto):
    pg.hotkey('ctrl', 'a') # selecionar tudo
    pg.press('backspace') #limpar
    pg.write(str(texto), interval=0.05)
    
def selecionar_status(st):
    mover_mouse("status")
    time.sleep(0.1)
    st = str(st).upper().strip()
    
    if st == "PENDENTE":
        pg.press('down')
    elif st ==  "BLOQUEADO":
        pg.press('down')
        pg.press('down')
    pg.press('enter')
    time.sleep(0.3)
    

def preencher_cadastro_mouse(r):
    
    mover_mouse("btn_novo")
    time.sleep(OPEN_MODAL)
    
    mover_mouse("nome")
    limpar_campos_e_escrever(r["nome"])
    
    mover_mouse("sobrenome")
    limpar_campos_e_escrever(r["sobrenome"])
    
    mover_mouse("cpf")
    limpar_campos_e_escrever(r["cpf"])
    
    mover_mouse("email")
    limpar_campos_e_escrever(r["email"])
    
    mover_mouse("telefone")
    limpar_campos_e_escrever(r["telefone"])
    
    mover_mouse("nascimento")
    limpar_campos_e_escrever(r["nascimento"])
    
    selecionar_status(r.get('status', 'ATIVO'))
             
    mover_mouse("observacao")
    limpar_campos_e_escrever(r["observacao"])
    
    mover_mouse("endereco")
    limpar_campos_e_escrever(r["endereco"])
    
    mover_mouse("btn_salvar")
    time.sleep(SAVE_REG)

def cadastrar_lote_coord(df, limit):
    open_portal(portal_url)
    for i, r in df.head(limit).iterrows():
        d = r.to_dict()
        preencher_cadastro_mouse(d)

In [70]:
cadastrar_lote_coord(df, limit=3)

## Acrônimos
- DOM: Document Object Model
- RPA: Robotic Process Automation
- CSV: Comma-Separated Values
- API: Application Programming Interface